# Phase 1: RAG Pipeline Setup (การเตรียมข้อมูลและการค้นหา)


In [ ]:
# !pip install -q langgraph langchain-core langchain-community langchain-openai langchain-chroma pydantic pandas

In [ ]:
import os
import getpass
import pandas as pd
from typing import List, Dict, Any, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END

# ตั้งค่าระบบตรวจรับ API Key จากผู้ใช้งาน
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

## 1. การเตรียมเอกสาร:

In [ ]:
doc_policy_content = """
# นโยบายความปลอดภัยและข้อบังคับทางการเงินประจำองค์กร (Corporate Policy v4.2)
ปรับปรุงล่าสุด: มีนาคม 2026

## หมวดที่ 1: ข้อบังคับและโครงสร้างการเคลมค่าเดินทางภายในประเทศ
- พนักงานระดับปฏิบัติการ (Staff) สามารถเบิกค่าเดินทางทางบก (เช่น รถแท็กซี่, รถไฟฟ้า) ได้ตามจริง แต่ห้ามเกิน 1,500 บาท ต่อวัน
- พนักงานระดับบริหาร (Manager ขึ้นไป) สามารถเบิกค่าเดินทางทางบกได้สูงสุดไม่เกิน 3,500 บาท ต่อวัน
- การเดินทางด้วยเครื่องบินชั้นประหยัด (Economy Class) อนุญาตเฉพาะกรณีที่ระยะทางจากสำนักงานใหญ่ไปยังปลายทางมากกว่า 300 กิโลเมตรเท่านั้น

## หมวดที่ 2: อัตราเบี้ยเลี้ยงการปฏิบัติงานนอกสถานที่ (Daily Allowance)
- การปฏิบัติงานต่างจังหวัด (ภายในประเทศ): พนักงานได้รับเบี้ยเลี้ยงคงที่อัตรา 600 บาท ต่อวัน (ไม่รวมค่าที่พัก)
- การปฏิบัติงานต่างประเทศ: พนักงานได้รับเบี้ยเลี้ยงคงที่อัตรา 2,500 บาท ต่อวัน

## หมวดที่ 3: นโยบายความปลอดภัยทางไซเบอร์และข้อมูล (Data & Cyber Security)
- ข้อมูลรหัสผ่านระบบและ API Key ขององค์กรทั้งหมด ถือเป็นข้อมูลความลับระดับสูงสุด (Strictly Confidential)
- ห้ามพนักงานทำการคัดลอก ส่งต่อ หรือแบ่งปัน รหัสผ่านหรือ API Key ผ่านแพลตฟอร์มแชตสาธารณะ (เช่น Line, Facebook Messenger) โดยเด็ดขาด
- การส่งมอบข้อมูลสำคัญประเภทนี้ จะต้องกระทำผ่านระบบรหัสลับภายในองค์กร (Internal Vault) เท่านั้น หากฝ่าฝืนมีโทษทางวินัยขั้นสูงสุด
"""

raw_docs = [Document(page_content=doc_policy_content, metadata={"source": "corporate_policy.txt"})]

# เลือกขนาด Chunk=450 และ Overlap=90 เพื่อให้ครอบคลุมบริบทเงื่อนไขของแต่ละหมวดหมู่ย่อยไม่ให้หลุดลอยแยกออกจากกัน
text_splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=90, separators=["\n\n", "\n", " "])
split_docs = text_splitter.split_documents(raw_docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = Chroma.from_documents(split_docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

## 2. การปรับปรุงการค้นหา (Query Transformation):


In [ ]:
class MultiQuerySchema(BaseModel):
    queries: List[str] = Field(description="สร้างคำถามใหม่ที่มีความหมายเดียวกันในแง่มุมที่หลากหลายจำนวน 3 ข้อความ")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", "คุณคือผู้เชี่ยวชาญด้านภาษาคอมพิวเตอร์ จงนำคำถามภาษาไทยของผู้ใช้มาขยายและเขียนใหม่ให้อยู่ในมุมมองที่หลากหลายและมีความหมายเหมือนเดิมจำนวน 3 ประโยคเพื่อนำไปค้นหาในฐานข้อมูลได้อย่างแม่นยำ"),
    ("human", "คำถามดั้งเดิม: {question}")
])

multi_query_chain = multi_query_prompt | llm.with_structured_output(MultiQuerySchema)

def run_transformed_rag(question: str) -> str:
    """ฟังก์ชันทำงาน RAG ควบคู่กับการใช้ Multi-Query Transformation"""
    print(f"   [Query Transformation] กำลังขยายมุมมองคำถาม...")
    transformed = multi_query_chain.invoke({"question": question})
    all_queries = [question] + transformed.queries
    
    # พิมพ์ล็อกคำถามที่ขยายมุมมองออกมาได้เพื่อตรวจสอบความโปร่งใส
    for idx, q_text in enumerate(transformed.queries):
         print(f"      -> คำถามย่อยที่ {idx+1}: {q_text}")
         
    # ระดมดึงข้อความจากคลังเวกเตอร์ด้วยคำถามทุกข้อ (รวบรวมแบบไม่มีตัวซ้ำ)
    unique_docs = {}
    for q in all_queries:
        docs = retriever.invoke(q)
        for d in docs:
            unique_docs[d.page_content] = d
            
    context_text = "\n\n".join(unique_docs.keys())
    
    # ส่งต่อบริบทที่ได้ให้ LLM สรุปคำตอบหลัก
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", "ตอบคำถามจากบริบทที่กำหนดให้เท่านั้นอย่างซื่อสัตย์ หากไม่มีให้แจ้งว่าหาไม่พบ:\n\n{context}"),
        ("human", "{question}")
    ])
    rag_chain = rag_prompt | llm | StrOutputParser()
    return rag_chain.invoke({"context": context_text, "question": question})

# Phase 2: Agent Architecture with Tools & Routing


## 1. การสร้าง Tools:


In [ ]:
def financial_calculator(expression: str) -> str:
    """เครื่องมือคำนวณสูตรคณิตศาสตร์ทางบัญชีและการเงิน รับตัวแปรเป็นข้อความสมการคณิตศาสตร์ดิบ เช่น 1500 * 5"""
    try:
        # ใช้ระบบประเมินผลสมการอย่างปลอดภัย
        result = eval(expression, {"__builtins__": None}, {})
        return f"ผลลัพธ์การคำนวณของโครงสร้างสมการ '{expression}' คือ {result}"
    except Exception as e:
        return f"Error: ไม่สามารถประมวลผลทางคณิตศาสตร์ได้เนื่องจากโครงสร้างผิดพลาด - {str(e)}"

## 2. การออกแบบ Agent Controller:


In [ ]:
class AgentState(BaseModel):
    question: str = Field(description="คำถามตั้งต้นของผู้ใช้")
    selected_route: str = Field(default="", description="เส้นทางที่ระบบเลือกเดินงาน")
    tool_result: str = Field(default="", description="ผลลัพธ์จากการเรียกใช้งานเครื่องมือภายนอก")
    final_response: str = Field(default="", description="คำตอบสรุปผลลัพธ์ด่านสุดท้าย")

class RouteClassifier(BaseModel):
    route: Literal["rag_policy", "financial_calc", "general_llm"] = Field(
        description="เลือก 'rag_policy' หากถามเรื่องสเปค เกณฑ์ ตัวเลขกฎ หรือนโยบายบริษัท, เลือก 'financial_calc' หากต้องการให้คำนวณสมการคณิตศาสตร์เป็นตัวเลข, เลือก 'general_llm' หากเป็นการชวนคุยหรือถามเรื่องทั่วไปภายนอกบริบทกฎองค์กร"
    )

def agent_router_logic(state: AgentState) -> Dict[str, Any]:
    """Node ด่านหน้า: ทำหน้าที่จำแนกแยกประเภทเจตนาคำถามส่งต่อให้ตรงสายงาน"""
    print("[Node] -> agent_router_logic (กำลังตรวจสอบคำสั่ง)")
    
    router_prompt = ChatPromptTemplate.from_messages([
        ("system", "คุณคือวิศวกรวิเคราะห์เจตนาคำสั่งซื้อและคำถามของพนักงานองค์กร จงจำแนกเพื่อส่งไปยังสถานีงานที่เหมาะสมที่สุดอย่างเคร่งครัด"),
        ("human", "{question}")
    ])
    
    router_chain = router_prompt | llm.with_structured_output(RouteClassifier)
    classification = router_chain.invoke({"question": state.question})
    print(f"   [Routing Analysis] เจตนาที่ตรวจพบ: {classification.route}")
    return {"selected_route": classification.route}

def execute_rag_node(state: AgentState) -> Dict[str, Any]:
    """Node สายงาน RAG: ส่งต่อไปประมวลผลดึงข้อมูลความรู้เชิงลึกด้วย Multi-Query"""
    print("[Node] -> execute_rag_node")
    response = run_transformed_rag(state.question)
    return {"final_response": response}

def execute_calc_node(state: AgentState) -> Dict[str, Any]:
    """Node สายงานคำนวณ: คัดแยกตัวเลขสร้างสมการและเรียกใช้ Financial Calculator"""
    print("[Node] -> execute_calc_node")
    
    # ใช้ LLM ช่วยแกะสกัดข้อความคำถามให้กลายเป็นสมการคณิตศาสตร์ดิบ
    calc_extractor = ChatPromptTemplate.from_template("จงสกัดข้อความนี้ให้กลายเป็นเพียงสมการคณิตศาสตร์ธรรมดาเพื่อส่งให้ฟังก์ชันคอมพิวเตอร์ประมวลผลต่อ เช่น 1500 * 3 โดยห้ามมีตัวอักษรอื่นปนเด็ดขาด:\n{question}")
    calc_chain = calc_extractor | llm | StrOutputParser()
    expression = calc_chain.invoke({"question": state.question}).strip()
    
    result = financial_calculator(expression)
    return {"tool_result": result}

def execute_general_node(state: AgentState) -> Dict[str, Any]:
    """Node สายงานทั่วไป: เรียกโมเดลตรงแก้ปัญหาคำถามจิปาถะทั่วไปนอกบริบทความรู้องค์กร"""
    print("[Node] -> execute_general_node")
    response = llm.invoke(state.question)
    return {"final_response": response.content}

def final_synthesizer_node(state: AgentState) -> Dict[str, Any]:
    """Node ด่านสุดท้าย: นำผลลัพธ์จากการคำนวณมาสังเคราะห์เรียบเรียงเป็นประโยคที่สละสลวย"""
    print("[Node] -> final_synthesizer_node")
    synthesize_prompt = ChatPromptTemplate.from_template("จงนำผลลัพธ์จากการรันเครื่องมือประมวลผลคณิตศาสตร์นี้: '{tool_result}' มาเรียบเรียงตอบคำถามเดิมของผู้ใช้: '{question}' ให้เป็นประโยคภาษาไทยที่สุภาพ เป็นธรรมชาติ และเข้าใจง่าย")
    synthesize_chain = synthesize_prompt | llm | StrOutputParser()
    response = synthesize_chain.invoke({"tool_result": state.tool_result, "question": state.question})
    return {"final_response": response}

# ==========================================
# 4. การประกอบทิศทางเงื่อนไขและการคอมไพล์เกราฟ
# ==========================================
def conditional_edge_decision(state: AgentState) -> Literal["rag_policy", "financial_calc", "general_llm"]:
    """ทำหน้าที่เป็นสะพานแยกสายงานอิงตามค่าสเตต selected_route"""
    return state.selected_route

agent_graph = StateGraph(AgentState)

# ลงทะเบียน Nodes เข้าเกราฟหลัก
agent_graph.add_node("agent_router_logic", agent_router_logic)
agent_graph.add_node("execute_rag_node", execute_rag_node)
agent_graph.add_node("execute_calc_node", execute_calc_node)
agent_graph.add_node("execute_general_node", execute_general_node)
agent_graph.add_node("final_synthesizer_node", final_synthesizer_node)

# เชื่อมเส้นทางจากจุดสตาร์ทเข้าสู่ด่านวิเคราะห์เจตนา
agent_graph.add_edge(START, "agent_router_logic")

# ตั้งค่ากลไกเงื่อนไขแยกทิศทางไปแต่ละสถานีงาน
agent_graph.add_conditional_edges(
    "agent_router_logic",
    conditional_edge_decision,
    {
        "rag_policy": "execute_rag_node",
        "financial_calc": "execute_calc_node",
        "general_llm": "execute_general_node"
    }
)

# จัดการทางเชื่อมปลายทางส่งกลับเข้าสู่จุดปิดงาน (END)
agent_graph.add_edge("execute_rag_node", END)
agent_graph.add_edge("execute_general_node", END)
agent_graph.add_edge("execute_calc_node", "final_synthesizer_node")
agent_graph.add_edge("final_synthesizer_node", END)

app = agent_graph.compile()

# Phase 3: Testing and Evaluation


## 1. การทดสอบ:


In [ ]:
### Test A (Pure RAG): คำถามที่ต้องตอบโดยอ้างอิงจากเอกสารภายในเท่านั้น (Proof of RAG)
### Test B (Tool Use): คำถามที่ต้องใช้ Tool (Proof of Agent)
### Test C (Integration/Routing): คำถามที่ต้องใช้ RAG Chain + Query Transformation ในการค้นหาคำตอบ
test_cases = [
    {"id": "Test A (Pure RAG)", "q": "ผู้จัดการสามารถเบิกค่าเดินทางทางบกต่อวันได้สูงสุดเท่าไหร่ และมีเงื่อนไขการขึ้นเครื่องบินอย่างไร"},
    {"id": "Test B (Tool Use)", "q": "พนักงานระดับปฏิบัติการออกไปปฏิบัติงานต่างจังหวัดเป็นเวลาทั้งหมด 5 วัน จะได้รับเงินเบี้ยเลี้ยงรวมทั้งสิ้นกี่บาท"},
    {"id": "Test C (Integration/Routing)", "q": "หากฉันเผลอส่ง API Key ไปให้เพื่อนในกลุ่ม Line จะถือว่าผิดกฎข้อบังคับของบริษัทในหมวดไหนและร้ายแรงไหม"}
]

execution_logs = []

print("="*60 + "\n เริ่มต้นกระบวนการทดสอบระบบประเมินผล AI Assistant \n" + "="*60)

for case in test_cases:
    print(f"\n[เริ่มรันกระบวนการ] รหัสทดสอบ: {case['id']}")
    print(f"คำถามผู้ใช้: '{case['q']}'")
    
    # ส่งคำถามทดสอบเข้าสู่กลไกสเตตเกราฟหลักของเอเจนต์
    state_output = app.invoke({"question": case['q']})
    
    execution_logs.append({
        "รหัสการทดสอบ": case['id'],
        "คำถามผู้ใช้ (Query)": case['q'],
        "เส้นทางที่เลือกเดิน (Selected Route)": state_output.get("selected_route"),
        "ผลลัพธ์สุดท้าย (Final Assistant Response)": state_output.get("final_response")
    })
    print(f"คำตอบจากระบบ: {state_output.get('final_response')}\n" + "-"*40)

In [ ]:
# การจัดทำตารางประเมินสรุปรายงานชิ้นงาน
print("\n" + "="*50 + "\n ตารางสรุปบันทึกการทำงานของระบบ AI Assistant \n" + "="*50)
df_final_report = pd.DataFrame(execution_logs)
pd.set_option('display.max_colwidth', None)
df_final_report

# Report & Reflection

## 1. เหตุผลที่เลือกใช้ Multi-Query Transformation และผลกระทบต่อความแม่นยำ
ในการพัฒนาระบบผู้ช่วยอัจฉริยะครั้งนี้ กระบวนการพัฒนาระบบสืบค้นข้อมูลได้ใช้เทคนิค Multi-Query Retrieval เข้ามาช่วยเพิ่มประสิทธิภาพในส่วนของ RAG Pipeline เนื่องจากพฤติกรรมการป้อนคำสั่งค้นหา (Search Queries) ของผู้ใช้งานภายในองค์กรมีความหลากหลายทางด้านคำศัพท์ (Lexical Variation) ตัวอย่างเช่น คำว่า "ผู้จัดการ" "หัวหน้าทีม" หรือ "พนักงานระดับบริหาร" ซึ่งมีความหมายในเชิงบริบทที่ใกล้เคียงกัน หากระบบส่งคำถามเริ่มต้นตรงเข้าสู่ฐานข้อมูลเวกเตอร์ (Vector Store) ทันที ค่าคะแนนระยะทางเวกเตอร์ (Distance Score) อาจมีความคลาดเคลื่อน ส่งผลให้การดึงชิ้นส่วนข้อความ (Retrieval Stage) ขาดความครบถ้วน

การนำ Large Language Model (LLM) เข้ามาช่วยขยายมุมมองและเรียบเรียงประโยคคำถามใหม่ (Query Rewriting) เพิ่มเติมจำนวน 3 ประโยคที่มีมิติทางภาษาหลากหลาย สามารถลดอัตราความผิดพลาดประเภทสืบค้นไม่พบ (False Negatives) ได้อย่างมีนัยสำคัญ ส่งผลให้ระบบตรวจค้นและดึงข้อมูลบริบทนโยบายจากฐานข้อมูล Chroma ได้อย่างหนาแน่น ครบถ้วน และช่วยให้โมเดลภาษาในส่วนการประมวลผลคำตอบ (Generation Stage) สรุปข้อมูลได้อย่างถูกต้องแม่นยำตรงตามข้อกำหนดจริงขององค์กร

## 2. สถาปัตยกรรมการตัดสินใจของเอเจนต์ (Agent Decision-Making Architecture) และกระบวนการทำงานภายใน
โครงสร้างระบบที่ออกแบบบนสถาปัตยกรรม LangGraph ดำเนินการบริหารจัดการผ่านศูนย์กลางข้อมูลสถานะกลาง (AgentState) โดยใช้กลไกควบคุมทิศทางแบบกำหนดเงื่อนไขที่แน่ชัด (Deterministic Node Routing) ซึ่งลำดับขั้นตอนการประมวลผลของระบบจากการทดสอบมีรายละเอียดดังต่อไปนี้:

- Test A (Pure RAG): ระบบประมวลผลด่านหน้า agent_router_logic ดำเนินการวิเคราะห์เจตนาของคำถามเมื่อจำแนกได้ว่าคำถามมีความเกี่ยวข้องกับระเบียบข้อบังคับ ระบบจะควบคุมทิศทางข้อมูลส่งต่อไปยัง execute_rag_node ทันทีเพื่อดึงข้อมูลความรู้จากเอกสารภายในองค์กร

- Test B (Tool Use - Intermediate Steps): คำถามที่ต้องการทราบผลรวมค่าเบี้ยเลี้ยงจากการปฏิบัติงานนอกสถานที่ ระบบส่วนควบคุมวิเคราะห์ว่าจำเป็นต้องใช้ทักษะทางคณิตศาสตร์ จึงส่งต่อคำสั่งไปยัง execute_calc_node ในขั้นตอนนี้เอเจนต์ดำเนินการสกัดข้อมูลบริบท (Information Extraction) เพื่อแปลงคำถามเชิงภาษาให้เป็นสมการทางคณิตศาสตร์ ($600 \times 5$) และเรียกใช้งานเครื่องมือ financial_calculator เมื่อได้รับผลลัพธ์ตัวเลข 3000 ข้อมูลจะถูกส่งต่อไปยัง final_synthesizer_node เพื่อเรียบเรียงเป็นประโยคสรุปผลลัพธ์ที่สละสลวยและตอบกลับผู้ใช้งานอย่างถูกต้อง

- Test C (Integration/Routing): สำหรับคำถามเชิงสถานการณ์จำลองเกี่ยวกับการส่งข้อมูลรหัสผ่านระบบบนแพลตฟอร์มสาธารณะ ระบบส่วนควบคุมจัดกลุ่มเจตนาคำถามให้อยู่ในหมวดหมู่ความปลอดภัยทางไซเบอร์ ข้อมูลจึงถูกส่งเข้าสู่กระบวนการ RAG ร่วมกับเทคนิค Multi-Query เพื่อสืบค้นเกณฑ์การลงโทษทางวินัยจากเอกสารอ้างอิง ก่อนนำมาสังเคราะห์เป็นคำเตือนและข้อควรปฏิบัติแจ้งแก่ผู้ใช้งานตามลำดับ